# WACV 2019 PCB Data Cleaning & SAM Point Extraction

Parses Pascal VOC XML annotations, filters out silkscreen text, runs SAM on component boxes, and exports polygon points.

In [ ]:
# Dependencies
!pip install -q opencv-python numpy pandas openpyxl matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import json
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
WACV_CLASSES = {
    'capacitor': {'kicad': 'Capacitor_SMD', 'footprint': 'Capacitor_SMD:C_0805_2012Metric', 'ref_prefix': 'C'},
    'electrolytic': {'kicad': 'Capacitor_THT', 'footprint': 'Capacitor_THT:CP_Radial_D6.3mm_P2.50mm', 'ref_prefix': 'C'},
    'resistor': {'kicad': 'Resistor_SMD', 'footprint': 'Resistor_SMD:R_0805_2012Metric', 'ref_prefix': 'R'},
    'ic': {'kicad': 'Package_SO', 'footprint': 'Package_SO:SOIC-8_3.9x4.9mm_P1.27mm', 'ref_prefix': 'U'},
    'transistor': {'kicad': 'Package_TO_SOT_SMD', 'footprint': 'Package_TO_SOT_SMD:SOT-23', 'ref_prefix': 'Q'},
    'diode': {'kicad': 'Diode_SMD', 'footprint': 'Diode_SMD:D_SOD-123', 'ref_prefix': 'D'},
    'connector': {'kicad': 'Connector', 'footprint': 'Connector_PinHeader_2.54mm:PinHeader_1x04_P2.54mm_Vert', 'ref_prefix': 'J'},
    'inductor': {'kicad': 'Inductor_SMD', 'footprint': 'Inductor_SMD:L_0805_2012Metric', 'ref_prefix': 'L'},
    'switch': {'kicad': 'Button_Switch_SMD', 'footprint': 'Button_Switch_SMD:SW_Push_SPST_NO_Alps_SKRK', 'ref_prefix': 'SW'},
    'button': {'kicad': 'Button_Switch_SMD', 'footprint': 'Button_Switch_SMD:SW_Push_SPST_NO_Alps_SKRK', 'ref_prefix': 'SW'},
    'led': {'kicad': 'LED_SMD', 'footprint': 'LED_SMD:LED_0805_2012Metric', 'ref_prefix': 'D'},
    'clock': {'kicad': 'Crystal', 'footprint': 'Crystal:Crystal_SMD_3225-4Pin_3.2x2.5mm', 'ref_prefix': 'Y'},
    'fuse': {'kicad': 'Fuse', 'footprint': 'Fuse:Fuse_1206_3216Metric', 'ref_prefix': 'F'},
    'transformer': {'kicad': 'Transformer_SMD', 'footprint': 'Transformer_SMD:Transformer_Bourns_SRF0703', 'ref_prefix': 'T'}
}
IGNORED_LABELS = {'text', 'pads', 'pins', 'unknown', 'test'}

In [ ]:
# Model checkpoint
SAM_CHECKPOINT = Path('sam_vit_b.pth')
sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CHECKPOINT))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)

In [ ]:
board_dir = Path('wacv_data/pcb_wacv_2019/ArduinoMega_Top')
xml_path = list(board_dir.glob('*.xml'))[0]
img_path = list(board_dir.glob('*.jpg'))[0]
output_dir = Path('wacv_sam_output')
output_dir.mkdir(parents=True, exist_ok=True)

img = cv2.imread(str(img_path))
h, w = img.shape[:2]
predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

tree = ET.parse(xml_path)
root = tree.getroot()
records, shapes, ref_counts = [], [], {}

for idx, obj in enumerate(root.findall('object'), 1):
    raw_name = obj.find('name').text if obj.find('name') is not None else ''
    first_word = raw_name.strip().strip('"').lower().split()[0] if raw_name else ''
    if first_word in IGNORED_LABELS:
        continue
    bnd = obj.find('bndbox')
    if bnd is None: continue
    x1, y1 = float(bnd.find('xmin').text), float(bnd.find('ymin').text)
    x2, y2 = float(bnd.find('xmax').text), float(bnd.find('ymax').text)
    
    masks, scores, _ = predictor.predict(box=np.array([x1, y1, x2, y2])[None, :], multimask_output=False)
    contours, _ = cv2.findContours(masks[0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: continue
    cnt = max(contours, key=cv2.contourArea)
    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True)
    pts = [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]
    
    info = WACV_CLASSES.get(first_word, {'kicad': 'Generic_Component', 'footprint': 'Package_SO:SOIC-8', 'ref_prefix': 'U'})
    pfx = info['ref_prefix']
    ref_counts[pfx] = ref_counts.get(pfx, 0) + 1
    ref_des = f'{pfx}{ref_counts[pfx]}'
    
    records.append({
        'board': board_dir.name, 'ref_des': ref_des, 'type': first_word,
        'kicad_footprint': info['footprint'], 'confidence': round(float(scores[0]), 3),
        'num_points': len(pts), 'points_compact': '; '.join([f'({p[0]},{p[1]})' for p in pts]),
        'points_json': json.dumps(pts)
    })

df = pd.DataFrame(records)
df.to_excel(output_dir / f'{board_dir.name}_sam_points.xlsx', index=False)
df.to_csv(output_dir / f'{board_dir.name}_sam_points.csv', index=False)
print(f'Processed {len(records)} components for {board_dir.name}.')
df.head()